In [11]:
from pathlib import Path
import mne
import pandas as pd


data = Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\Data\cap-sleep-database-1.0.0\n1.edf")
txt_path = r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\Data\cap-sleep-database-1.0.0\n1.txt"

raw = mne.io.read_raw_edf(data, preload=True, verbose=False)

print(raw.info["sfreq"])
print(raw.ch_names[:10])

C:\Users\carlo\AppData\Local\Temp\ipykernel_920\3252767969.py:9: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\3252767969.py:9: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\3252767969.py:9: RuntimeWarning: Highpass cutoff frequency 10.0 is greater than lowpass cutoff frequency 3.0, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(data, preload=True, verbose=False)


512.0
['ROC-LOC', 'LOC-ROC', 'F2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F1-F3', 'F3-C3', 'C3-P3', 'P3-O1']


In [12]:
!pip install h5py


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import importlib
import functions
import script

importlib.reload(functions)
importlib.reload(script)

from script import process_patient_to_hdf5
from functions import *


In [14]:
df = pd.read_csv(
    txt_path,
    sep='\t',
    header=None,
    names=["Sleep Stage", "Position", "Time [hh:mm:ss]", "Event", "Duration[s]", "Location"],
    skiprows=22
)

df["Duration[s]"] = pd.to_numeric(df["Duration[s]"], errors="coerce")
df = df[df["Event"].astype(str).str.startswith("SLEEP-", na=False)].copy()
df = df[df["Duration[s]"] == 30].copy()
df = df[df["Sleep Stage"].astype(str).isin(["W", "R", "S1", "S2", "S3", "S4"])].reset_index(drop=True)

print("Rows after filtering:", len(df))
print(df["Sleep Stage"].value_counts())

df.head()

Rows after filtering: 1140
Sleep Stage
S2    508
R     239
S4    186
S3    135
W      39
S1     33
Name: count, dtype: int64


,Sleep Stage,Position,Time [hh:mm:ss],Event,Duration[s],Location
0,W,Unknown Position,22:09:33,SLEEP-S0,30,ROC-LOC
1,W,Unknown Position,22:10:03,SLEEP-S0,30,ROC-LOC
2,W,Unknown Position,22:10:33,SLEEP-S0,30,ROC-LOC
3,W,Unknown Position,22:11:03,SLEEP-S0,30,ROC-LOC
4,W,Unknown Position,22:11:33,SLEEP-S0,30,ROC-LOC


In [19]:
process_patient_to_hdf5(
    raw=raw,
    patient_id="n01",
    group="control",
    df=df,
    output_dir=r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files"
)

BP DF length: 1140
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5


In [20]:
import h5py

h5_path = r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5"

with h5py.File(h5_path, "r") as f:
    print(list(f.keys()))                 # should show ['n01']
    print(list(f["n01"]["epochs"].keys())[:5])  # first epochs

['n01']
['epoch_0000', 'epoch_0001', 'epoch_0002', 'epoch_0003', 'epoch_0004']


In [21]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd

patient_id = "n01"
candidate_paths = [
    Path("../HDF files") / f"{patient_id}.h5",
    Path("HDF files") / f"{patient_id}.h5",
    Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5"),
]

path = next((p for p in candidate_paths if p.exists()), candidate_paths[-1])
print("Exists:", path.exists())
print("Path:", path.resolve())


Exists: True
Path: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5


In [22]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

patient_id = "n01"

candidate_paths = [
    Path("../HDF files") / f"{patient_id}.h5",
    Path("HDF files") / f"{patient_id}.h5",
    Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files") / f"{patient_id}.h5",
]

path = next((p for p in candidate_paths if p.exists()), candidate_paths[-1])

print("Exists:", path.exists())
print("Path:", path.resolve())

rows = []

with h5py.File(path, "r") as f:
    print("\nTop level keys:", list(f.keys()))

    patient = f[patient_id]
    print("Attrs:", dict(patient.attrs))
    print("Subgroups:", list(patient.keys()))

    print("\nGlobal counts")
    for key in ["n_epochs_all", "n_epochs_rem", "n_epochs_nrem", "n_valid_hep_epochs"]:
        if key in patient:
            print(f"{key}:", patient[key][()])

    print("\nGlobal means")
    for key in [
        "mean_delta_power_all",
        "mean_delta_power_REM",
        "mean_delta_power_NREM",
        "mean_hep_all",
        "mean_hep_REM",
        "mean_hep_NREM",
    ]:
        if key in patient:
            print(f"{key}:", patient[key][()])

    if "hep_waveform_derived" in patient:
        wf = patient["hep_waveform_derived"]
        print("\nHEP waveform-derived keys:", list(wf.keys()))
        for key in [
            "hep_peak_latency_all",
            "hep_peak_latency_REM",
            "hep_peak_latency_NREM",
            "hep_auc_all",
            "hep_auc_REM",
            "hep_auc_NREM",
        ]:
            if key in wf:
                print(f"{key}:", wf[key][()])

    if "correlations" in patient:
        corr = patient["correlations"]
        print("\nGlobal 30s correlation keys:", list(corr.keys()))
        for key in sorted(corr.keys()):
            print(f"{key}:", corr[key][()])

    epoch_keys = list(patient["epochs"].keys())
    print("\nN epochs:", len(epoch_keys))
    print("First 5 epochs:", epoch_keys[:5])

    ep = patient["epochs"][epoch_keys[0]]
    stage = ep["sleep_stage"][()]
    if isinstance(stage, bytes):
        stage = stage.decode()

    print("\nFirst epoch summary")
    print("start_time:", ep["start_time"][()])
    print("end_time:", ep["end_time"][()])
    print("sleep_stage:", stage)
    print("delta:", ep["eeg_bandpower"]["delta"][()])
    print("hr_mean_bpm:", ep["hrv"]["hr_mean_bpm"][()])
    print("cpc ratio:", ep["cpc"]["ratio"][()])
    print("pearson_1s_r:", ep["hep"]["pearson_1s_r"][()])
    print("hep_30s:", ep["hep"]["hep_30s"][()])
    print("delta_30s:", ep["hep"]["delta_30s"][()])
    print("waveform mean shape:", ep["waveform"]["mean"]["signal"].shape)

    for ep_name in epoch_keys:
        ep = patient["epochs"][ep_name]

        stage = ep["sleep_stage"][()]
        if isinstance(stage, bytes):
            stage = stage.decode()

        row = {
            "epoch": ep_name,
            "start": ep["start_time"][()],
            "end": ep["end_time"][()],
            "stage": stage,
            "delta": ep["eeg_bandpower"]["delta"][()],
            "theta": ep["eeg_bandpower"]["theta"][()],
            "alpha": ep["eeg_bandpower"]["alpha"][()],
            "beta": ep["eeg_bandpower"]["beta"][()],
            "gamma": ep["eeg_bandpower"]["gamma"][()],
            "n_beats": ep["hrv"]["n_beats"][()],
            "hr_mean_bpm": ep["hrv"]["hr_mean_bpm"][()],
            "rmssd_ms": ep["hrv"]["rmssd_ms"][()],
            "sdnn_ms": ep["hrv"]["sdnn_ms"][()],
            "pnn50_pct": ep["hrv"]["pnn50_pct"][()],
            "hfc": ep["cpc"]["hfc"][()],
            "lfc": ep["cpc"]["lfc"][()],
            "cpc_ratio": ep["cpc"]["ratio"][()],
            "pearson_1s_r": ep["hep"]["pearson_1s_r"][()],
            "pearson_1s_p": ep["hep"]["pearson_1s_p"][()],
            "spearman_1s_r": ep["hep"]["spearman_1s_r"][()],
            "spearman_1s_p": ep["hep"]["spearman_1s_p"][()],
            "hep_30s": ep["hep"]["hep_30s"][()],
            "delta_30s": ep["hep"]["delta_30s"][()],
            "waveform_mean_len": len(ep["waveform"]["mean"]["signal"][()]),
            "waveform_max_len": len(ep["waveform"]["max"]["signal"][()]),
        }

        rows.append(row)

df_check = pd.DataFrame(rows)

print("\nPer-epoch table preview")
display(df_check.head())

print("\nPer-epoch describe")
display(df_check.describe(include="all"))

print("\nNaN count per column:")
print(df_check.isna().sum())

print("\nEpoch durations:", df_check["end"].sub(df_check["start"]).unique())
print("Stages:", sorted(df_check["stage"].dropna().unique()))
print("Rows with HEP 1s missing:", df_check["pearson_1s_r"].isna().sum())
print("Rows with CPC missing:", df_check["cpc_ratio"].isna().sum())
print("Rows with HEP 30s missing:", df_check["hep_30s"].isna().sum())
print("Rows with delta 30s missing:", df_check["delta_30s"].isna().sum())

print("\nFirst 10 compact rows")
display(
    df_check[
        ["epoch", "stage", "delta", "hr_mean_bpm", "cpc_ratio", "pearson_1s_r", "hep_30s", "delta_30s"]
    ].head(10)
)


Exists: True
Path: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5

Top level keys: ['n01']
Attrs: {'group': 'control', 'sfreq': np.float64(512.0)}
Subgroups: ['correlations', 'epochs', 'hep_waveform_derived', 'mean_delta_power_NREM', 'mean_delta_power_REM', 'mean_delta_power_all', 'mean_hep_NREM', 'mean_hep_REM', 'mean_hep_all', 'n_epochs_all', 'n_epochs_nrem', 'n_epochs_rem', 'n_valid_hep_epochs']

Global counts
n_epochs_all: 1140
n_epochs_rem: 239
n_epochs_nrem: 862
n_valid_hep_epochs: 1139

Global means
mean_delta_power_all: 0.5117540458329375
mean_delta_power_REM: 0.41849868374872534
mean_delta_power_NREM: 0.536621252801761
mean_hep_all: 3.1046229615620358
mean_hep_REM: 1.8415371563684195
mean_hep_NREM: 3.409241939297796

HEP waveform-derived keys: ['auc_window_end_s', 'auc_window_start_s', 'hep_auc_NREM', 'hep_auc_REM', 'hep_auc_all', 'hep_peak_latency_NREM', 'hep_peak_latency_REM', 'hep_peak_latency_all', 'hep_

,epoch,start,end,stage,delta,theta,alpha,beta,gamma,n_beats,...,lfc,cpc_ratio,pearson_1s_r,pearson_1s_p,spearman_1s_r,spearman_1s_p,hep_30s,delta_30s,waveform_mean_len,waveform_max_len
0,epoch_0000,0,30,Wake,1.580909e-11,9.583595e-12,1.969461e-12,6.620830e-12,2.073184e-12,40,...,53375.424047,0.153245,-0.265845,0.610626,-0.257143,0.622787,9.005253,0.438457,409,409
1,epoch_0001,30,60,Wake,5.037948e-11,6.067240e-12,9.792978e-13,5.932491e-12,1.971509e-12,45,...,53375.424047,0.153245,0.832192,0.080409,0.900000,0.037386,22.744214,0.771154,409,409
2,epoch_0002,60,90,Wake,3.078252e-11,7.327169e-12,1.915875e-12,2.794727e-12,5.358076e-13,43,...,53375.424047,0.153245,0.386912,0.448592,0.428571,0.396501,1.728724,0.709993,409,409
3,epoch_0003,90,120,Wake,3.450004e-11,8.378086e-12,9.988140e-13,2.756804e-12,4.326136e-13,41,...,53375.424047,0.153245,NaN,NaN,NaN,NaN,6.203347,0.733008,409,409
4,epoch_0004,120,150,Wake,3.101924e-11,5.835149e-12,1.530392e-12,3.488126e-12,7.569416e-13,41,...,53375.424047,0.153245,0.872454,0.325055,0.500000,0.666667,9.559751,0.727641,409,409



Per-epoch describe


,epoch,start,end,stage,delta,theta,alpha,beta,gamma,n_beats,...,lfc,cpc_ratio,pearson_1s_r,pearson_1s_p,spearman_1s_r,spearman_1s_p,hep_30s,delta_30s,waveform_mean_len,waveform_max_len
count,1140,1140.000000,1140.000000,1140,1.140000e+03,1.140000e+03,1.140000e+03,1.140000e+03,1.140000e+03,1140.000000,...,1140.000000,1140.000000,1030.000000,1030.000000,1030.000000,1030.000000,1139.000000,1140.000000,1140.0,1140.0
unique,1140,NaN,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,epoch_0000,NaN,NaN,N2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,508,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,17085.000000,17115.000000,NaN,8.507335e-12,3.671196e-12,9.214270e-13,1.431515e-12,7.122019e-14,35.779825,...,5906.498665,0.231653,0.248328,0.310899,0.207125,0.360208,3.104623,0.511754,409.0,409.0
std,NaN,9877.018781,9877.018781,NaN,1.131620e-11,6.206152e-12,1.439182e-12,1.999521e-12,2.953100e-13,2.277449,...,25415.276889,0.271478,0.283070,0.295811,0.278925,0.297986,3.462135,0.128139,0.0,0.0
min,NaN,0.000000,30.000000,NaN,6.490442e-13,7.063661e-13,2.994118e-13,4.237171e-13,2.296001e-14,30.000000,...,151.698445,0.029999,-0.997144,0.000001,-1.000000,0.000000,0.519606,0.172987,409.0,409.0
25%,NaN,8542.500000,8572.500000,NaN,2.796728e-12,1.947638e-12,5.902267e-13,9.024591e-13,3.110719e-14,34.000000,...,520.633451,0.095560,0.095041,0.043380,0.055840,0.077650,1.361109,0.427514,409.0,409.0
50%,NaN,17085.000000,17115.000000,NaN,5.085986e-12,2.868499e-12,7.886579e-13,1.216477e-12,3.513030e-14,36.000000,...,947.053787,0.158755,0.278406,0.208726,0.214262,0.296413,1.863954,0.501974,409.0,409.0
75%,NaN,25627.500000,25657.500000,NaN,1.038379e-11,4.288806e-12,1.057395e-12,1.587981e-12,4.093563e-14,38.000000,...,1631.999095,0.263007,0.432631,0.519709,0.381389,0.613567,3.092407,0.600362,409.0,409.0



NaN count per column:
epoch                  0
start                  0
end                    0
stage                  0
delta                  0
theta                  0
alpha                  0
beta                   0
gamma                  0
n_beats                0
hr_mean_bpm            0
rmssd_ms               0
sdnn_ms                0
pnn50_pct              0
hfc                    0
lfc                    0
cpc_ratio              0
pearson_1s_r         110
pearson_1s_p         110
spearman_1s_r        110
spearman_1s_p        110
hep_30s                1
delta_30s              0
waveform_mean_len      0
waveform_max_len       0
dtype: int64

Epoch durations: [30]
Stages: ['N1', 'N2', 'N3', 'REM', 'Wake']
Rows with HEP 1s missing: 110
Rows with CPC missing: 0
Rows with HEP 30s missing: 1
Rows with delta 30s missing: 0

First 10 compact rows


,epoch,stage,delta,hr_mean_bpm,cpc_ratio,pearson_1s_r,hep_30s,delta_30s
0,epoch_0000,Wake,1.580909e-11,82.088945,0.153245,-0.265845,9.005253,0.438457
1,epoch_0001,Wake,5.037948e-11,89.896952,0.153245,0.832192,22.744214,0.771154
2,epoch_0002,Wake,3.078252e-11,86.221324,0.153245,0.386912,1.728724,0.709993
3,epoch_0003,Wake,3.450004e-11,82.328332,0.153245,NaN,6.203347,0.733008
4,epoch_0004,Wake,3.101924e-11,83.701401,0.153245,0.872454,9.559751,0.727641
5,epoch_0005,Wake,1.450020e-12,77.731868,0.039295,0.410192,2.303551,0.259223
6,epoch_0006,Wake,3.554569e-12,78.859589,0.039295,-0.169239,2.092632,0.453416
7,epoch_0007,Wake,6.490442e-13,75.822127,0.039295,0.392227,1.897268,0.180428
8,epoch_0008,Wake,7.261734e-13,76.758021,0.039295,0.597804,2.391744,0.205650
9,epoch_0009,Wake,1.321111e-12,79.700860,0.039295,0.503013,4.272935,0.352489


In [ ]:
import gc
from pathlib import Path

import mne
import pandas as pd
from IPython.display import display

from functions import load_cap_hypnogram, pick_core_channels, prepare_sleep_stage_dataframe
from script import process_patient_to_hdf5

base_dir = Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD")
data_dir = base_dir / "Data" / "cap-sleep-database-1.0.0"
output_dir = base_dir / "HDF files"

controls = ["n1", "n2", "n3", "n4", "n5", "n9", "n10", "n11", "ins1", "ins2", "ins3", "ins4", "ins5", "ins6", "ins7", "ins8", "ins9"]
rbd_patients = ["rbd2", "rbd3", "rbd5", "rbd7", "rbd8", "rbd9", "rbd10", "rbd11", "rbd12", "rbd13", "rbd17", "rbd18", "rbd19", "rbd21", "rbd22"]

patients = [(p, "control") for p in controls] + [(p, "rbd") for p in rbd_patients]
min_valid_h5_bytes = 1024

def normalize_patient_id(file_stem: str) -> str:
    if file_stem.startswith("n") and file_stem[1:].isdigit():
        return f"n{int(file_stem[1:]):02d}"
    if file_stem.startswith("rbd") and file_stem[3:].isdigit():
        return f"rbd{int(file_stem[3:]):02d}"
    return file_stem

pending_patients = []
saved_patients = []

for file_stem, group in patients:
    patient_id = normalize_patient_id(file_stem)
    h5_path = output_dir / f"{patient_id}.h5"

    if h5_path.exists() and h5_path.stat().st_size >= min_valid_h5_bytes:
        saved_patients.append((file_stem, patient_id, group))
    else:
        pending_patients.append((file_stem, patient_id, group))

print(f"Already saved: {len(saved_patients)}")
print(f"Still to process: {len(pending_patients)}")
print([patient_id for _, patient_id, _ in pending_patients])

results = []

for file_stem, patient_id, group in pending_patients:
    edf_path = data_dir / f"{file_stem}.edf"
    txt_path = data_dir / f"{file_stem}.txt"
    h5_path = output_dir / f"{patient_id}.h5"

    print(f"\n=== Processing {file_stem} -> {patient_id} ({group}) ===")

    raw = None
    raw_core = None
    df = None
    df_prepared = None

    if not edf_path.exists():
        print(f"Missing EDF: {edf_path}")
        results.append({"file_stem": file_stem, "patient_id": patient_id, "group": group, "status": "missing_edf"})
        continue

    if not txt_path.exists():
        print(f"Missing TXT: {txt_path}")
        results.append({"file_stem": file_stem, "patient_id": patient_id, "group": group, "status": "missing_txt"})
        continue

    if h5_path.exists() and h5_path.stat().st_size < min_valid_h5_bytes:
        h5_path.unlink(missing_ok=True)

    try:
        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        raw_core = pick_core_channels(raw)

        print("Loaded channels:", raw_core.ch_names)
        print("sfreq:", raw_core.info["sfreq"])
        print("n_times:", raw_core.n_times)

        df = load_cap_hypnogram(txt_path)
        df_prepared = prepare_sleep_stage_dataframe(df, epoch_len=30)

        print("Rows after filtering:", len(df_prepared))
        print(df_prepared["Sleep Stage"].value_counts())

        process_patient_to_hdf5(
            raw=raw_core,
            patient_id=patient_id,
            group=group,
            df=df,
            output_dir=output_dir
        )

        print(f"Saved: {h5_path}")
        results.append({"file_stem": file_stem, "patient_id": patient_id, "group": group, "status": "ok"})

    except Exception as e:
        print(f"FAILED: {e}")

        if h5_path.exists() and h5_path.stat().st_size < 1024:
            h5_path.unlink(missing_ok=True)

        results.append({"file_stem": file_stem, "patient_id": patient_id, "group": group, "status": f"error: {e}"})

    finally:
        del raw_core
        del raw
        del df
        del df_prepared
        gc.collect()

results_df = pd.DataFrame(results, columns=["file_stem", "patient_id", "group", "status"])
display(results_df)

print("\nStatus count:")
if results_df.empty:
    print("No pending patients. All requested HDF files already exist.")
else:
    print(results_df["status"].value_counts())



=== Processing n1 -> n01 (control) ===
Reading 0 ... 17725439  =      0.000 ... 34619.998 secs...


C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Highpass cutoff frequency 10.0 is greater than lowpass cutoff frequency 3.0, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 17725440
Rows after filtering: 1140
Sleep Stage
S2    508
R     239
S4    186
S3    135
W      39
S1     33
Name: count, dtype: int64
BP DF length: 1140
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5
Saved: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n01.h5

=== Processing n2 -> n02 (control) ===
Reading 0 ... 22579199  =      0.000 ... 44099.998 secs...


C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 22579200
Rows after filtering: 0
Series([], Name: count, dtype: int64)
FAILED: No valid 30 s sleep-stage epochs remain after filtering the hypnogram.

=== Processing n3 -> n03 (control) ===
Reading 0 ... 16927231  =      0.000 ... 33060.998 secs...


...

C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_920\1906055072.py:51: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 18109440
Rows after filtering: 1087
Sleep Stage
S2    374
R     300
W     185
S4    111
S3     72
S1     45
Name: count, dtype: int64
BP DF length: 1087
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\rbd22.h5
Saved: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\rbd22.h5


,file_stem,patient_id,group,status
0,n1,n01,control,ok
1,n2,n02,control,error: No valid 30 s sleep-stage epochs remain...
2,n3,n03,control,error: No valid 30 s sleep-stage epochs remain...
3,n4,n04,control,error: Missing required channels: ['ECG1-ECG2']
4,n5,n05,control,ok
5,n9,n09,control,"error: Missing required channels: ['F4-C4', 'E..."
6,n10,n10,control,ok
7,n11,n11,control,ok
8,ins1,ins1,control,error: No valid 30 s sleep-stage epochs remain...
9,ins2,ins2,control,error: No valid 30 s sleep-stage epochs remain...



Status count:
status
ok                                                                               16
error: No valid 30 s sleep-stage epochs remain after filtering the hypnogram.    14
error: Missing required channels: ['ECG1-ECG2']                                   1
error: Missing required channels: ['F4-C4', 'ECG1-ECG2']                          1
Name: count, dtype: int64


: 

In [2]:
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
raw_core = pick_core_channels(raw)
raw_core.load_data()


Reading 0 ... 15544319  =      0.000 ... 30359.998 secs...


C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\583526940.py:1: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\583526940.py:1: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


<RawEDF | rbd21.edf, 2 x 15544320 (30360.0 s), ~237.2 MiB, data loaded>

In [3]:
import gc
import importlib
from pathlib import Path

import mne
import pandas as pd
from IPython.display import display

import functions
import script

importlib.reload(functions)
importlib.reload(script)

from functions import load_cap_hypnogram, pick_core_channels, prepare_sleep_stage_dataframe
from script import process_patient_to_hdf5

base_dir = Path(r"C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD")
data_dir = base_dir / "Data" / "cap-sleep-database-1.0.0"
output_dir = base_dir / "HDF files"

remaining_controls = ["n2", "n3", "n4", "ins1", "ins2", "ins3", "ins4", "ins5", "ins7"]
remaining_rbd = ["rbd3", "rbd5", "rbd11", "rbd12", "rbd13", "rbd21"]

patients = [(p, "control") for p in remaining_controls] + [(p, "rbd") for p in remaining_rbd]

def normalize_patient_id(file_stem: str) -> str:
    if file_stem.startswith("n") and file_stem[1:].isdigit():
        return f"n{int(file_stem[1:]):02d}"
    if file_stem.startswith("rbd") and file_stem[3:].isdigit():
        return f"rbd{int(file_stem[3:]):02d}"
    return file_stem

results = []

for file_stem, group in patients:
    patient_id = normalize_patient_id(file_stem)
    edf_path = data_dir / f"{file_stem}.edf"
    txt_path = data_dir / f"{file_stem}.txt"
    h5_path = output_dir / f"{patient_id}.h5"

    print(f"\n=== Processing {file_stem} -> {patient_id} ({group}) ===")

    raw = None
    raw_core = None
    df = None
    df_prepared = None

    try:
        if not edf_path.exists():
            raise FileNotFoundError(f"Missing EDF: {edf_path}")

        if not txt_path.exists():
            raise FileNotFoundError(f"Missing TXT: {txt_path}")

        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        raw_core = pick_core_channels(raw)
        raw_core.load_data()

        print("Loaded channels:", raw_core.ch_names)
        print("sfreq:", raw_core.info["sfreq"])
        print("n_times:", raw_core.n_times)

        df = load_cap_hypnogram(txt_path)
        df_prepared = prepare_sleep_stage_dataframe(
            df,
            epoch_len=30,
            raw_duration_s=raw_core.n_times / raw_core.info["sfreq"],
        )

        print("Rows after filtering:", len(df_prepared))
        print(df_prepared["Sleep Stage"].value_counts())

        process_patient_to_hdf5(
            raw=raw_core,
            patient_id=patient_id,
            group=group,
            df=df,
            output_dir=output_dir,
        )

        print(f"Saved: {h5_path}")
        results.append({
            "file_stem": file_stem,
            "patient_id": patient_id,
            "group": group,
            "status": "ok",
        })

    except Exception as e:
        print(f"FAILED: {e}")
        results.append({
            "file_stem": file_stem,
            "patient_id": patient_id,
            "group": group,
            "status": f"error: {e}",
        })

    finally:
        del raw_core
        del raw
        del df_prepared
        del df
        gc.collect()

results_df = pd.DataFrame(results)
display(results_df)

print("\nStatus count:")
print(results_df["status"].value_counts())



=== Processing n2 -> n02 (control) ===
Reading 0 ... 22579199  =      0.000 ... 44099.998 secs...


C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\1508443596.py:56: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\1508443596.py:56: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 22579200
Rows after filtering: 999
Sleep Stage
S2    367
R     151
W     143
S1    141
S4    114
S3     83
Name: count, dtype: int64
BP DF length: 999
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n02.h5
Saved: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n02.h5

=== Processing n3 -> n03 (control) ===
Reading 0 ... 16927231  =      0.000 ... 33060.998 secs...


C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\1508443596.py:56: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
C:\Users\carlo\AppData\Local\Temp\ipykernel_23336\1508443596.py:56: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)


Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 16927232
Rows after filtering: 999
Sleep Stage
S2    347
R     188
S4    167
W     136
S3    112
S1     49
Name: count, dtype: int64
BP DF length: 999
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n03.h5
Saved: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\n03.h5

=== Processing n4 -> n04 (control) ===
Reading 0 ... 7147999  =      0.000 ... 35739.995 secs...
Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 200.0
n_times: 7148000
Rows after filtering: 979
Sleep Stage
S2    401
W     223
R     198
S4     76
S3     63
S1     18
Name: count, dtype: int64
FAILED: lowpass frequency [ 50.625 100.75 ] must be less than Nyquist (100.0)

=== Processing ins1 -> ins1 (control) ===
Reading 0 ... 12287999  =      0.000 ... 47999.996 secs...


...

Loaded channels: ['F4-C4', 'ECG1-ECG2']
sfreq: 512.0
n_times: 15544320
Rows after filtering: 983
Sleep Stage
S2    345
W     253
S3    132
R     128
S4     68
S1     57
Name: count, dtype: int64


c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
c:\Users\carlo\miniforge3\envs\dtu02452\Lib\site-packag

BP DF length: 983
Saved → C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\rbd21.h5
Saved: C:\Users\carlo\OneDrive - Universidade de Lisboa\Documents\GitHub\heart-lung-brain-coupling-for-RBD\HDF files\rbd21.h5


,file_stem,patient_id,group,status
0,n2,n02,control,ok
1,n3,n03,control,ok
2,n4,n04,control,error: lowpass frequency [ 50.625 100.75 ] mus...
3,ins1,ins1,control,ok
4,ins2,ins2,control,ok
5,ins3,ins3,control,ok
6,ins4,ins4,control,ok
7,ins5,ins5,control,ok
8,ins7,ins7,control,ok
9,rbd3,rbd03,rbd,ok



Status count:
status
ok                                                                              14
error: lowpass frequency [ 50.625 100.75 ] must be less than Nyquist (100.0)     1
Name: count, dtype: int64
